# **Symmetry Breaking in the CC-MUCP via Demand-Aware Lexicographic Ordering**

This notebook studies symmetry reduction for the Combined Cycle Min-Up/Min-Down
Unit Commitment Problem (CC-MUCP) under a component-based formulation.

> See `ccmucp_formulation.md` for the complete mathematical formulation, parameter
> tables, and detailed results analysis.

---

### **Symmetry Structure**

When all GTs within each package are identical, the CC-MUCP exhibits
intra-package permutation symmetry. Within each package $k$, the
$n_{gt}$ GTs are interchangeable: any permutation $\sigma \in S_{n_{gt}}$
of the GT indices yields a feasible solution of equal cost.

The symmetry group is the direct product
$$H = S_{n_{\mathrm{gt}}} \times \cdots \times S_{n_{\mathrm{gt}}}\quad (K\text{ copies}),$$
of order $|H| = (n_{\mathrm{gt}}!)^K$. For $n_{\mathrm{gt}} = 3$ and
$K = 2$, this gives $|H| = (3!)^2 = 36$.

To break this symmetry, we apply a demand-aware lexicographic ordering:
time periods are ranked by decreasing demand, and within each package GTs
are ordered so that those covering more high-demand periods receive lower
indices. This is enforced by prefix-sum constraints in the model builder.
Packages are treated as distinguishable, so no package-level ordering
is imposed.

### **Example: Two-Package CC-MUCP Instance**

We consider a CC plant with $K = 2$ packages, each consisting of
$n_{gt} = 3$ gas turbines and one steam turbine ($3 \times 1$ configuration),
over a weekly horizon of $T = 21$ periods structured as three load levels
per day (Low, Mid, High).

In combined-cycle mode, GTs operate at lower costs than in open-cycle mode,
reflecting the efficiency gain from heat recovery. The ST runs on residual
exhaust heat with no variable cost, and has higher start-up cost and longer
minimum up/down times reflecting steam cycle thermal inertia. Coupling
parameters enforce that at least two GTs must be online before the ST can
start, and must remain so after it shuts down.

> The parameter tables and full instance data are in `ccmucp_formulation.md`.

In [1]:
using JuMP, HiGHS, JLD2

In [2]:
# Instance parameters 
function instance_parameters()
    K     = 2
    n_gt  = 3
    T     = 21

    # GT parameters (identical across all GTs)
    Pmax_OC = 150.0;  Pmin_OC =  50.0
    Pmax_CC = 145.0;  Pmin_CC =  60.0
    cf_OC   = 844.0;  cp_OC   =  37.1
    cf_CC   = 549.0;  cp_CC   =  24.1
    c0_g    = 25_000.0
    L_g     = 2;      ell_g   = 1

    # ST parameters (identical across all packages)
    Pmin_v  =  20.0;  alpha   =  40.0
    cf_v    =  42.0;  cp_v    =   0.0
    c0_v    = 15_000.0
    L_v     = 3;      ell_v   = 2

    # Coupling parameters
    m_k      = 2
    delta_up = 1
    delta_dn = 1

    # Demand profile: weekly 3-level (Low, Mid, High) for 7 days
    DAYS   = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
    LEVELS = ["L","M","H"]
    T = length(DAYS) * length(LEVELS)
    D = [420.,620.,900., 400.,600.,880., 430.,640.,920.,
         410.,630.,910., 450.,700.,1000.,520.,680.,820.,
         500.,650.,780.]

    return K, n_gt, T, D, DAYS, LEVELS,
           Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
           cf_OC, cp_OC, cf_CC, cp_CC, c0_g, L_g, ell_g,
           Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
           m_k, delta_up, delta_dn
end

instance_parameters (generic function with 1 method)

In [3]:
# Ranks periods from highest to lowest demand
function demand_period_ranking(D)
    return sortperm(D, rev=true)
end


# Extracts binary commitment arrays from a solved model
function read_solution(x_g, x_v, K, n_gt, T)
    return round.(Int, value.(x_g)), round.(Int, value.(x_v))
end


# Sorts GTs within each package by demand coverage (canonical orbit rep)
function reference_schedule(gt_plan, st_plan, K, n_gt, T, demand_rank)
    gt_new = copy(gt_plan)
    st_new = copy(st_plan)

    for k in 1:K
        prefix_sums = [cumsum(gt_plan[k, g, demand_rank]) for g in 1:n_gt]
        perm = sortperm(prefix_sums, rev=true)
        for g in 1:n_gt
            gt_new[k, g, :] = gt_plan[k, perm[g], :]
        end
    end

    return gt_new, st_new
end


# Checks whether a solution already satisfies the GT ordering
function is_reference(gt_plan, K, n_gt, T, demand_rank)
    for k in 1:K, g in 1:(n_gt-1)
        for R in 1:T
            if sum(gt_plan[k, g,   demand_rank[r]] for r in 1:R) <
               sum(gt_plan[k, g+1, demand_rank[r]] for r in 1:R)
                return false
            end
        end
    end
    return true
end


# Prints the commitment schedule for all GTs and STs
function show_schedule(gt_plan, st_plan, K, n_gt, T; label="")
    label != "" && println(label)
    for k in 1:K
        println("  Package ", k, ":")
        for g in 1:n_gt
            println("    GT ", g, ": ", gt_plan[k,g,:],
                    "  (hours on: ", sum(gt_plan[k,g,:]), ")")
        end
        println("    ST  : ", st_plan[k,:],
                "  (hours on: ", sum(st_plan[k,:]), ")")
    end
end

show_schedule (generic function with 1 method)

In [4]:
# Builds the CC-MUCP MILP model (symmetry_breaking: :none or :symmetry)
function build_ccmucp_model(K, n_gt, T, D,
                              Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
                              cf_OC, cp_OC, cf_CC, cp_CC, c0_g,
                              L_g, ell_g,
                              Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
                              m_k, delta_up, delta_dn;
                              symmetry_breaking = :none,
                              demand_rank       = nothing,
                              target_cost       = nothing)

    model = Model(HiGHS.Optimizer)
    set_silent(model)

    # Variables
    @variable(model, x_g[1:K, 1:n_gt, 1:T], Bin)
    @variable(model, u_g[1:K, 1:n_gt, 1:T], Bin)
    @variable(model, y_OC[1:K, 1:n_gt, 1:T], Bin)
    @variable(model, y_CC[1:K, 1:n_gt, 1:T], Bin)
    @variable(model, p_OC[1:K, 1:n_gt, 1:T] >= 0)
    @variable(model, p_CC[1:K, 1:n_gt, 1:T] >= 0)
    @variable(model, x_v[1:K, 1:T], Bin)
    @variable(model, u_v[1:K, 1:T], Bin)
    @variable(model, p_v[1:K, 1:T] >= 0)

    # Objective
    @objective(model, Min,
        sum(cf_OC * y_OC[k,g,t] + cf_CC * y_CC[k,g,t] +
            c0_g  * u_g[k,g,t]  +
            cp_OC * p_OC[k,g,t] + cp_CC * p_CC[k,g,t]
            for k in 1:K, g in 1:n_gt, t in 1:T) +
        sum(cf_v * x_v[k,t] + c0_v * u_v[k,t] + cp_v * p_v[k,t]
            for k in 1:K, t in 1:T))

    # GT constraints
    for k in 1:K, g in 1:n_gt
        @constraint(model, u_g[k,g,1] >= x_g[k,g,1])
        for t in 2:T
            @constraint(model, u_g[k,g,t] >= x_g[k,g,t] - x_g[k,g,t-1])
        end
        for t in 1:T
            for s in 1:min(L_g - 1, T - t)
                @constraint(model, x_g[k,g,t+s] >= u_g[k,g,t])
            end
            @constraint(model, x_g[k,g,t] == y_OC[k,g,t] + y_CC[k,g,t])
            @constraint(model, p_OC[k,g,t] >= Pmin_OC * y_OC[k,g,t])
            @constraint(model, p_OC[k,g,t] <= Pmax_OC * y_OC[k,g,t])
            @constraint(model, p_CC[k,g,t] >= Pmin_CC * y_CC[k,g,t])
            @constraint(model, p_CC[k,g,t] <= Pmax_CC * y_CC[k,g,t])
        end
        for t in 2:T
            for s in 1:min(ell_g - 1, T - t)
                @constraint(model,
                    x_g[k,g,t+s] <= 1 - (x_g[k,g,t-1] - x_g[k,g,t]))
            end
        end
    end

    # ST constraints
    for k in 1:K
        @constraint(model, u_v[k,1] >= x_v[k,1])
        for t in 2:T
            @constraint(model, u_v[k,t] >= x_v[k,t] - x_v[k,t-1])
        end
        for t in 1:T
            for s in 1:min(L_v - 1, T - t)
                @constraint(model, x_v[k,t+s] >= u_v[k,t])
            end
            @constraint(model, p_v[k,t] >= Pmin_v * x_v[k,t])
            @constraint(model, p_v[k,t] <= alpha * sum(y_CC[k,g,t] for g in 1:n_gt))
        end
        for t in 2:T
            for s in 1:min(ell_v - 1, T - t)
                @constraint(model,
                    x_v[k,t+s] <= 1 - (x_v[k,t-1] - x_v[k,t]))
            end
        end
    end

    # Coupling constraints CC1-CC5
    for k in 1:K, t in 1:T
        @constraint(model,
            sum(x_g[k,g,t] for g in 1:n_gt) >= m_k * x_v[k,t])
        for g in 1:n_gt
            @constraint(model, y_CC[k,g,t] <= x_v[k,t])
            @constraint(model, y_OC[k,g,t] <= 1 - x_v[k,t])
        end
        if t <= delta_up
            @constraint(model, u_v[k,t] == 0)
        else
            @constraint(model,
                m_k * u_v[k,t] <= sum(x_g[k,g,t - delta_up] for g in 1:n_gt))
        end
        if t >= 2
            for s in 0:(delta_dn - 1)
                if t + s <= T
                    for g in 1:n_gt
                        @constraint(model,
                            x_g[k,g,t+s] >= x_v[k,t-1] - x_v[k,t])
                    end
                end
            end
        end
    end

    # Global demand
    for t in 1:T
        @constraint(model,
            sum(p_OC[k,g,t] + p_CC[k,g,t] for k in 1:K, g in 1:n_gt) +
            sum(p_v[k,t] for k in 1:K) >= D[t])
    end

    # Symmetry-breaking ordering
    if symmetry_breaking == :symmetry
        @assert demand_rank !== nothing
        for k in 1:K, g in 1:(n_gt-1)
            for R in 1:T
                @constraint(model,
                    sum(x_g[k,g,   demand_rank[r]] for r in 1:R) >=
                    sum(x_g[k,g+1, demand_rank[r]] for r in 1:R))
            end
        end
    end

    # Target cost for enumeration
    if target_cost !== nothing
        @constraint(model,
            sum(cf_OC*y_OC[k,g,t] + cf_CC*y_CC[k,g,t] +
                c0_g*u_g[k,g,t]   +
                cp_OC*p_OC[k,g,t] + cp_CC*p_CC[k,g,t]
                for k in 1:K, g in 1:n_gt, t in 1:T) +
            sum(cf_v*x_v[k,t] + c0_v*u_v[k,t] + cp_v*p_v[k,t]
                for k in 1:K, t in 1:T) == target_cost)
    end

    return model, x_g, u_g, y_OC, y_CC, p_OC, p_CC, x_v, u_v, p_v
end

build_ccmucp_model (generic function with 1 method)

In [5]:
# Enumerates all optimal solutions by adding no-good cuts
function enumerate_ccmucp(K, n_gt, T, D,
                            Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
                            cf_OC, cp_OC, cf_CC, cp_CC, c0_g,
                            L_g, ell_g,
                            Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
                            m_k, delta_up, delta_dn,
                            best_cost;
                            symmetry_breaking = :none,
                            demand_rank       = nothing,
                            max_solutions     = 500)

    solutions = Vector{Tuple{Array{Int,3}, Matrix{Int}}}()

    for _ in 1:max_solutions
        m, x_g, u_g, y_OC, y_CC, p_OC, p_CC, x_v, u_v, p_v =
            build_ccmucp_model(K, n_gt, T, D,
                Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
                cf_OC, cp_OC, cf_CC, cp_CC, c0_g, L_g, ell_g,
                Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
                m_k, delta_up, delta_dn;
                symmetry_breaking = symmetry_breaking,
                demand_rank       = demand_rank,
                target_cost       = best_cost)

        for (prev_gt, prev_st) in solutions
            @constraint(m,
                sum(1 - x_g[k,g,t]
                    for k in 1:K, g in 1:n_gt, t in 1:T
                    if prev_gt[k,g,t] == 1; init=0) +
                sum(x_g[k,g,t]
                    for k in 1:K, g in 1:n_gt, t in 1:T
                    if prev_gt[k,g,t] == 0; init=0) +
                sum(1 - x_v[k,t]
                    for k in 1:K, t in 1:T
                    if prev_st[k,t] == 1; init=0) +
                sum(x_v[k,t]
                    for k in 1:K, t in 1:T
                    if prev_st[k,t] == 0; init=0) >= 1)
        end

        optimize!(m)
        termination_status(m) != OPTIMAL && break
        gt_plan, st_plan = read_solution(x_g, x_v, K, n_gt, T)
        push!(solutions, (gt_plan, st_plan))
    end

    return solutions
end

enumerate_ccmucp (generic function with 1 method)

In [6]:
# Groups solutions into orbits using the canonical representative
function orbit_decomposition(solutions, K, n_gt, T, demand_rank)
    reps = Vector{Tuple{Array{Int,3}, Matrix{Int}}}()
    orbit_sizes = Int[]
    sol_to_rep_idx = Vector{Int}(undef, length(solutions))

    for (i, (gt, st)) in enumerate(solutions)
        gt_canon, st_canon = reference_schedule(
            gt, st, K, n_gt, T, demand_rank)
        found = false
        for (j, (rgt, rst)) in enumerate(reps)
            if gt_canon == rgt && st_canon == rst
                orbit_sizes[j] += 1
                sol_to_rep_idx[i] = j
                found = true
                break
            end
        end
        if !found
            push!(reps, (gt_canon, st_canon))
            push!(orbit_sizes, 1)
            sol_to_rep_idx[i] = length(reps)
        end
    end

    return reps, orbit_sizes, sol_to_rep_idx
end

orbit_decomposition (generic function with 1 method)

### **Instance Setup**

In [7]:
K, n_gt, T, D, DAYS, LEVELS,
    Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
    cf_OC, cp_OC, cf_CC, cp_CC, c0_g, L_g, ell_g,
    Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
    m_k, delta_up, delta_dn = instance_parameters()

demand_rank = demand_period_ranking(D)

period_label(t) = DAYS[(t-1)÷3 + 1] * "-" * LEVELS[(t-1)%3 + 1]

println("Topology: K = ", K, " packages | ", n_gt, "x1 configuration | T = ", T, " periods")
println("Max system capacity: ", K * (n_gt * Pmax_CC + n_gt * alpha), " MW")
println("Peak demand:    ", maximum(D), " MW at t = ", argmax(D),
        " (", period_label(argmax(D)), ")")
println("Minimum demand: ", minimum(D), " MW at t = ", argmin(D),
        " (", period_label(argmin(D)), ")")

println("\nDemand profile:")
for t in 1:T
    bar = repeat("\u2588", round(Int, D[t]/25))
    println("  t = ", lpad(t, 2), " | ",
        rpad(period_label(t), 10),
        " | ", lpad(Int(D[t]), 4), " MW | ", bar)
end

println("\nRanking pi: ", demand_rank)

Topology: K = 2 packages | 3x1 configuration | T = 21 periods
Max system capacity: 1110.0 MW
Peak demand:    1000.0 MW at t = 15 (Fri-H)
Minimum demand: 400.0 MW at t = 4 (Tue-L)

Demand profile:
  t =  1 | Mon-L      |  420 MW | █████████████████
  t =  2 | Mon-M      |  620 MW | █████████████████████████
  t =  3 | Mon-H      |  900 MW | ████████████████████████████████████
  t =  4 | Tue-L      |  400 MW | ████████████████
  t =  5 | Tue-M      |  600 MW | ████████████████████████
  t =  6 | Tue-H      |  880 MW | ███████████████████████████████████
  t =  7 | Wed-L      |  430 MW | █████████████████
  t =  8 | Wed-M      |  640 MW | ██████████████████████████
  t =  9 | Wed-H      |  920 MW | █████████████████████████████████████
  t = 10 | Thu-L      |  410 MW | ████████████████
  t = 11 | Thu-M      |  630 MW | █████████████████████████
  t = 12 | Thu-H      |  910 MW | ████████████████████████████████████
  t = 13 | Fri-L      |  450 MW | ██████████████████
  t = 14 | Fri-M     

In [8]:
m, x_g, u_g, y_OC, y_CC, p_OC, p_CC, x_v, u_v, p_v = 
    build_ccmucp_model(K, n_gt, T, D,
        Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
        cf_OC, cp_OC, cf_CC, cp_CC, c0_g, L_g, ell_g,
        Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
        m_k, delta_up, delta_dn;
        symmetry_breaking = :none)
optimize!(m)
best_cost = round(objective_value(m), digits=2)
println("Optimal cost: \$$best_cost")

Optimal cost: $489683.0


### **Enumeration**

In [9]:
# Baseline (no symmetry breaking)
println("Enumerating with :none...")
sols_base = enumerate_ccmucp(K, n_gt, T, D,
    Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
    cf_OC, cp_OC, cf_CC, cp_CC, c0_g, L_g, ell_g,
    Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
    m_k, delta_up, delta_dn, best_cost;
    symmetry_breaking = :none)
println("  Found $(length(sols_base)) solutions")

Enumerating with :none...
  Found 108 solutions


In [10]:
# With symmetry breaking
println("Enumerating with :symmetry...")
sols_sym = enumerate_ccmucp(K, n_gt, T, D,
    Pmin_OC, Pmax_OC, Pmin_CC, Pmax_CC,
    cf_OC, cp_OC, cf_CC, cp_CC, c0_g, L_g, ell_g,
    Pmin_v, alpha, cf_v, cp_v, c0_v, L_v, ell_v,
    m_k, delta_up, delta_dn, best_cost;
    symmetry_breaking = :symmetry,
    demand_rank = demand_rank)
println("  Found $(length(sols_sym)) solutions")

Enumerating with :symmetry...
  Found 8 solutions


### **Orbit Analysis**

In [11]:
reps, orbit_sizes, _ = orbit_decomposition(
    sols_base, K, n_gt, T, demand_rank)

println("Orbit decomposition:")
println("  Total solutions: $(length(sols_base))")
println("  Number of orbits: $(length(reps))")
println("  Orbit sizes: $(sort(orbit_sizes, rev=true))")
println("  Check sum: $(sum(orbit_sizes))")

Orbit decomposition:
  Total solutions: 108
  Number of orbits: 8
  Orbit sizes: [18, 18, 18, 18, 9, 9, 9, 9]
  Check sum: 108


In [12]:
# Save results
results_dict = Dict(
    "instance" => (K=K, n_gt=n_gt, T=T),
    "demand_rank" => demand_rank,
    "best_cost" => best_cost,
    "sols_base" => sols_base,
    "sols_sym" => sols_sym,
    "reps" => reps,
    "orbit_sizes" => orbit_sizes)
save("results/ccmucp_results.jld2", results_dict)
println("Results saved to results/ccmucp_results.jld2")

Results saved to results/ccmucp_results.jld2
